# Yearline Universe Engine — V13 MVP (V13.0 + V13.1)

**Universe-first, sector-aware, ticker-agnostic** statistical context engine.

This notebook *orchestrates only*: it loads a universe config and calls the
reusable modules in `src/yearline_universe/`. There is no analysis logic in the
notebook itself — the same `run_ticker_pipeline` runs every ticker.

> Educational research only. Not financial advice. Evidence overlay; never trades.


## 1. Setup

In [ ]:
import sys, json, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(REPO / 'src'))
import pandas as pd
from yearline_universe import (
    load_universe_config, run_ticker_pipeline, run_universe_pipeline,
    export_single_ticker_context, validate_ticker_sanity,
    ml_feature_leakage_audit, hazard_feature_leakage_audit,
)
from yearline_universe.dashboard import build_cross_sectional_dashboard
from yearline_universe.pooling import build_pooled_context
from yearline_universe.context_export import write_json
CACHE = REPO / 'data' / 'price_cache'
print('yearline_universe ready')

## 2. Load the universe
The top-level unit is the **universe**, not a single ticker.

In [ ]:
uni = load_universe_config(REPO / 'config' / 'universe_mega_cap_ai_infra.yaml')
print('universe:', uni.universe_name)
print('tickers :', list(uni.symbols))
print('sectors :', uni.sectors())
print('peers   :', uni.peer_groups())
pd.DataFrame([{'ticker': t.ticker, 'sector': t.sector, 'peer_group': t.peer_group, 'role': t.role} for t in uni.tickers])

## 3. V13.1 — generic single-ticker pipeline
The identical `run_ticker_pipeline(ticker_config, universe_config)` runs MSFT,
AAPL and NVDA. No ticker-specific branching. (Cache covers these three; other
configured tickers need live data via `provider='auto'`.)

In [ ]:
TICKERS = ['MSFT', 'AAPL', 'NVDA']
results = {}
for sym in TICKERS:
    results[sym] = run_ticker_pipeline(uni.get_ticker(sym), uni, cache_dir=str(CACHE), provider='cache')
    r = results[sym]
    print(f"{sym}: status={r.status} events={len(r.canonical_events)} "
          f"engine={r.latest_context.get('active_engine_context',{}).get('active_engine')}")

### Per-ticker statistical context envelopes

In [ ]:
rows = []
for sym, r in results.items():
    e = export_single_ticker_context(r)
    rows.append({
        'ticker': e['ticker'], 'as_of': e['as_of'], 'sector': e['sector'],
        'active_engine': e['active_engine_context']['active_engine'],
        'mode_state': e['active_engine_context']['mode_state'],
        'distance_to_ma250_pct': e['repair_retry_context']['distance_to_ma250_pct'],
        'p_retry_within_40d_gated': e['retry_hazard_context']['p_retry_within_40d'],
        'trend_state': e['post_confirmation_trend_context']['trend_state'],
        'trend_quality': e['post_confirmation_trend_context']['trend_quality_score'],
        'research_hint': e['option_overlay_research_hint']['research_hint'],
    })
pd.DataFrame(rows)

In [ ]:
# Full envelope for one ticker (repo-ready JSON)
print(json.dumps(export_single_ticker_context(results['MSFT']), indent=2)[:1800])

## 4. V13.4 preview — cross-sectional dashboard
Run the whole universe as a batch (V13.2). Tickers without cached/live data fail
in isolation and are recorded in the run manifest rather than killing the run.

In [ ]:
universe_result = run_universe_pipeline(uni, cache_dir=str(CACHE), provider='cache')
print('run manifest:', universe_result.run_manifest['n_ok'], '/', universe_result.run_manifest['n_tickers'], 'ok')
build_cross_sectional_dashboard(universe_result)

## 5. V13.3 preview — peer-group / sector pooling (basic)

In [ ]:
display(build_pooled_context(universe_result.ticker_results, group_by='peer_group'))
build_pooled_context(universe_result.ticker_results, group_by='sector')

## 6. Validation — sanity gate + anti-leakage audit

In [ ]:
for sym, r in results.items():
    s = validate_ticker_sanity(r)
    print(f"{sym}: sanity {'PASS' if s['passed'] else 'FAIL'} ({s['n_checks']} checks)")
ml_feature_leakage_audit().head(8)

## 7. Export repo-ready context
Per-ticker envelopes, the universe bundle, and the run manifest.

In [ ]:
tdir = REPO / 'exports' / 'ticker_contexts'
udir = REPO / 'exports' / 'universe_contexts'
for sym, r in results.items():
    if r.status == 'ok':
        write_json(export_single_ticker_context(r), tdir / f'{sym}_statistical_context.json')
write_json(universe_result.universe_context_bundle, udir / f'{uni.universe_name}_bundle.json')
write_json(universe_result.run_manifest, udir / f'{uni.universe_name}_run_manifest.json')
print('exports written to', REPO / 'exports')

---
*V13.0 + V13.1 MVP. V12 is the frozen research reference. Pooling (V13.3),
dashboard plots (V13.4), universe replay (V13.6) and sector calibration (V13.7)
are planned. Educational research only — not financial advice.*